# 04. Presentation Layer for the Power BI Dashboard

This notebook:

1. Builds `presentation.flights_dashboard`, a fact table with business-friendly column names and a few derived fields.
2. Builds `presentation.dim_airport`, one row per airport.
3. Validates both tables against staging and against the figures from Notebook 03.
4. Documents how to connect Power BI Desktop to them.

**Scope.** Database objects for reporting. The dashboard itself is built by hand in Power BI Desktop and saved in `dashboard/`.

**Prerequisites.** `staging.stg_bts_flights` from Notebook 02 and the `.env` in the project root.

A separate `presentation` schema keeps the dashboard independent from the internal structure of the ETL tables. If staging changes, only this notebook has to be adjusted.

# 1. Environment setup

Same connection setup as Notebooks 02 and 03. This notebook reads from staging and writes to the new schema.

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import logging
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    logger.info(f"Connected to database: {result.scalar()}")

2026-10-27 10:08:42,641 - INFO - Connected to database: bts_flights_db


# 2. Fact table

I materialize the table with `CREATE TABLE AS SELECT`. A view would recompute against 7M rows on every refresh while I design visuals, and rebuilding the table only means re-running this cell.

Compared with staging, the ETL fields (`row_hash`, `loaded_at`) are dropped and a few columns are renamed to what Power BI should display. Two fields are derived:

- `on_time_status` with four values (On Time, Delayed, Cancelled, Diverted), the same breakdown as Notebook 03, Section 6. A categorical field is easier to use in slicers than rebuilding the logic in DAX. Diverted flights need their own branch: their `arr_del15` is null, and my first version of this `CASE` sent them to `'On Time'` through the `ELSE`.
- `sched_dep_hour`, the scheduled departure hour (`crs_dep_time / 100`).

`taxi_out` stays in the table for the airport congestion page.

In [2]:
fact_table_ddl = """
CREATE SCHEMA IF NOT EXISTS presentation;

DROP TABLE IF EXISTS presentation.flights_dashboard;

CREATE TABLE presentation.flights_dashboard AS
SELECT
    flight_date,
    flight_year,
    flight_month,
    quarter,
    day_of_week,

    reporting_airline   AS carrier_code,
    tail_number,

    origin,
    origin_city_name,
    origin_state,
    dest,
    dest_city_name,
    dest_state,

    crs_dep_time,
    (crs_dep_time / 100) AS sched_dep_hour,
    dep_delay,
    crs_arr_time,
    arr_delay,
    arr_del15,

    cancelled,
    cancellation_code,
    diverted,
    distance,
    taxi_out,

    carrier_delay,
    weather_delay,
    nas_delay,
    security_delay,
    late_aircraft_delay,

    CASE
        WHEN cancelled = 1 THEN 'Cancelled'
        WHEN diverted = 1 THEN 'Diverted'
        WHEN arr_del15 = 1 THEN 'Delayed'
        ELSE 'On Time'
    END AS on_time_status

FROM staging.stg_bts_flights;

CREATE INDEX IF NOT EXISTS idx_flights_dashboard_date ON presentation.flights_dashboard (flight_date);
CREATE INDEX IF NOT EXISTS idx_flights_dashboard_carrier ON presentation.flights_dashboard (carrier_code);
CREATE INDEX IF NOT EXISTS idx_flights_dashboard_origin ON presentation.flights_dashboard (origin);
"""

with engine.connect() as conn:
    conn.execute(text(fact_table_ddl))
    conn.commit()
    logger.info("presentation.flights_dashboard created.")

2026-10-06 13:19:30,685 - INFO - presentation.flights_dashboard created.


# 3. Airport dimension

The fact table already has city and state for both ends of each flight. A separate `dim_airport` adds a single list of airports, one row each, which is what a slicer needs when the user picks "an airport". It's also where latitude and longitude would go if I add a map, without repeating them on 7M rows.

Codes come from both `origin` and `dest` through a `UNION`, because some airports could appear on only one side. `UNION` already removes repeated rows, so no `DISTINCT` is needed, and `airport_code` is declared as the primary key: if a code ever appeared with two spellings of its city, the table creation would fail instead of silently breaking the relationship in Power BI.

In [3]:
dim_airport_ddl = """
DROP TABLE IF EXISTS presentation.dim_airport;

CREATE TABLE presentation.dim_airport AS
SELECT
    airport_code,
    city_name,
    state
FROM (
    SELECT origin AS airport_code, origin_city_name AS city_name, origin_state AS state
    FROM staging.stg_bts_flights
    UNION
    SELECT dest, dest_city_name, dest_state
    FROM staging.stg_bts_flights
) AS combined_airports;

ALTER TABLE presentation.dim_airport ADD PRIMARY KEY (airport_code);
"""

with engine.connect() as conn:
    conn.execute(text(dim_airport_ddl))
    conn.commit()
    logger.info("presentation.dim_airport created.")

2026-10-27 10:10:36,265 - INFO - presentation.dim_airport created.


In [4]:
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM presentation.dim_airport")).scalar()
    logger.info(f"Unique airports: {count}")

2026-10-27 10:10:41,736 - INFO - Unique airports: 348


348 airports is plausible for U.S. domestic commercial aviation, but a plausible count doesn't prove that each code appears once. The `UNION` works on the full (code, city, state) triple, so a code with two spellings of its city would produce two rows and break the one-to-many relationship in Power BI. The primary key already rules that out, but I keep this query as a visible check.

In [5]:
with engine.connect() as conn:
    df_dup_airports = pd.read_sql(text("""
        SELECT a.airport_code, a.city_name, a.state
        FROM presentation.dim_airport AS a
        JOIN (
            SELECT airport_code
            FROM presentation.dim_airport
            GROUP BY airport_code
            HAVING COUNT(*) > 1
        ) AS dup USING (airport_code)
        ORDER BY a.airport_code;
    """), conn)

logger.info(f"Duplicated airport codes: {df_dup_airports['airport_code'].nunique()}")
df_dup_airports

2026-10-27 10:11:09,725 - INFO - Duplicated airport codes: 0


,airport_code,city_name,state


The result is empty: each of the 348 airport codes maps to a single city and state, so `airport_code` can act as the key on the "one" side of the relationship.

# 4. Validation

Row counts against staging, nulls in the key dashboard fields, and the `on_time_status` breakdown. The derived status is where a logic bug would hide, so I reconcile it with the figures Notebook 03 computed independently.

In [6]:
with engine.connect() as conn:
    fact_count = conn.execute(text("SELECT COUNT(*) FROM presentation.flights_dashboard")).scalar()
    staging_count = conn.execute(text("SELECT COUNT(*) FROM staging.stg_bts_flights")).scalar()
    logger.info(f"presentation.flights_dashboard: {fact_count:,} | staging.stg_bts_flights: {staging_count:,}")

    null_check = conn.execute(text("""
        SELECT
            COUNT(*) FILTER (WHERE carrier_code IS NULL) AS null_carrier,
            COUNT(*) FILTER (WHERE flight_date IS NULL) AS null_date,
            COUNT(*) FILTER (WHERE on_time_status IS NULL) AS null_status
        FROM presentation.flights_dashboard
    """)).fetchone()
    logger.info(f"Unexpected nulls: {null_check}")

    status_breakdown = conn.execute(text("""
        SELECT on_time_status, COUNT(*) AS n
        FROM presentation.flights_dashboard
        GROUP BY on_time_status
        ORDER BY n DESC
    """)).fetchall()
    logger.info("on_time_status breakdown:")
    for status, n in status_breakdown:
        logger.info(f"  {status}: {n:,}")

2026-10-06 13:21:21,079 - INFO - presentation.flights_dashboard: 7,079,061 | staging.stg_bts_flights: 7,079,061
2026-10-06 13:21:22,005 - INFO - Unexpected nulls: (0, 0, 0)
2026-10-06 13:21:23,289 - INFO - on_time_status breakdown:
2026-10-06 13:21:23,289 - INFO -   On Time: 5,515,281
2026-10-06 13:21:23,289 - INFO -   Delayed: 1,449,966
2026-10-06 13:21:23,295 - INFO -   Cancelled: 96,315
2026-10-06 13:21:23,298 - INFO -   Diverted: 17,499


7,079,061 rows, matching staging, with no nulls in the key fields. The status breakdown reproduces Notebook 03, Section 6 exactly: 5,515,281 on time, 1,449,966 delayed, 96,315 cancelled and 17,499 diverted. On Time plus Diverted gives 5,532,780, the figure the earlier three-category version had labelled "On Time", which confirms what the fix changed and that nothing else moved.

# 5. Power BI connection guide

These steps happen in Power BI Desktop, outside the notebook.

### Connecting

1. **Get Data** → **PostgreSQL database**.
2. **Server:** `localhost:5432` (your `DB_HOST:DB_PORT`). **Database:** `bts_flights_db`.
3. Choose **Import** mode. The fact table is row-level (7M rows), but Power BI's columnar compression handles that size comfortably, and Import keeps the visuals fast while designing.
4. Use `DB_USER` and `DB_PASSWORD` from `.env` when prompted.
5. In the Navigator, select `presentation.flights_dashboard` and `presentation.dim_airport`, then **Load**.

### Relationships

`dim_airport[airport_code]` has to relate to both `origin` and `dest`, and Power BI allows only one active relationship between two tables. Two options:

- Duplicate the dimension in Power Query as `dim_airport_origin` and `dim_airport_dest`, each with its own active relationship.
- Keep one `dim_airport` related to `origin` only, and use `dest_city_name` / `dest_state` from the fact table for destination filters.

The second is simpler and enough if the visuals mostly filter by departure airport.

### Suggested pages

1. **Overview:** scheduled flights, % delayed, % cancelled, `on_time_status` breakdown, monthly volume.
2. **Carriers:** delay rate vs. median delay when delayed, cancellations by carrier.
3. **Causes:** delay minutes by cause, with a carrier slicer. The Pareto with the cumulative share is in Notebook 03, Section 4.
4. **Airports and routes:** taxi-out by origin airport, worst routes, `sched_dep_hour` profile.

# 6. Conclusions

The presentation layer is ready for Power BI. `presentation.flights_dashboard` holds all 7,079,061 flights with readable column names, a four-category `on_time_status` that reconciles exactly with Notebook 03, and the hour and taxi-out fields the operational pages need. `presentation.dim_airport` lists 348 airports with no duplicated codes.

The diverted-flight branch was the fix that mattered here. Without it, 17,499 diverted flights would have shown up as on time in every visual.